# Seguridad de la API gRPC

SAIF aplicado al servicio real `tp_mlops2.grpc_api`, que corre en `localhost:50051`
dentro de Docker Compose. El interceptor de autenticacion vive en
`src/tp_mlops2/grpc_api/server.py` (`AuthInterceptor`) y esta activo en el contenedor
`grpc-api` todo el tiempo: no es un servidor de prueba levantado para este notebook.

In [1]:
import grpc

from tp_mlops2.grpc_api import scoring_pb2, scoring_pb2_grpc
from tp_mlops2.grpc_api.client import SAMPLE_REQUEST
from tp_mlops2.auth import crear_token

channel = grpc.insecure_channel("localhost:50051")
stub = scoring_pb2_grpc.ScoringStub(channel)
request = scoring_pb2.PredictRequest(**SAMPLE_REQUEST)

## 1. Sin credenciales: UNAUTHENTICATED

El `AuthInterceptor` corta la llamada antes de que llegue al modelo.

In [2]:
try:
    stub.Predict(request)
    print("no deberia haber llegado aca")
except grpc.RpcError as e:
    print("sin token:", e.code(), "(esperado UNAUTHENTICATED)")

sin token: StatusCode.UNAUTHENTICATED (esperado UNAUTHENTICATED)


## 2. Con JWT valido: predice sobre el modelo real

In [3]:
token_ok = crear_token("alan", "analista")
md = (("authorization", f"Bearer {token_ok}"),)

respuesta = stub.Predict(request, metadata=md)
print("con JWT valido:", respuesta.predicted_load_mw, "MW (modelo", respuesta.model_version, ")")

con JWT valido: 30694.077109585807 MW (modelo random_forest_demanda@production )


## 3. Token vencido: tambien rechazado

In [4]:
import datetime
import jwt

from tp_mlops2.auth import JWT_SECRET

token_vencido = jwt.encode(
    {
        "sub": "alan", "rol": "analista",
        "exp": datetime.datetime.now(datetime.UTC) - datetime.timedelta(minutes=1),
    },
    JWT_SECRET, algorithm="HS256",
)

try:
    stub.Predict(request, metadata=(("authorization", f"Bearer {token_vencido}"),))
    print("no deberia haber llegado aca")
except grpc.RpcError as e:
    print("token vencido:", e.code(), "(esperado UNAUTHENTICATED)")

channel.close()

token vencido: StatusCode.UNAUTHENTICATED (esperado UNAUTHENTICATED)


## Reflexion

El interceptor verifica el JWT antes de que la llamada llegue al `ScoringServicer`, asi
que una credencial invalida nunca toca el modelo. Reutiliza el mismo `tp_mlops2.auth` que
REST y GraphQL: un token emitido con `crear_token` vale en las tres capas, lo que es
justamente la coherencia de controles que pide SAIF -no una solucion distinta por
protocolo.